Inspired by the Medium Blog written by Mayur Pethani on AES(Automated Essay Scoring):

https://medium.com/analytics-vidhya/automated-essay-scoring-kaggle-competition-end-to-end-project-implementation-part-1-b75a043903c4

Here, my goal is to show the Pytorch way of implementing a Deep Customizable LSTM Architecture where we can easily add as many LSTM layers with different hidden_dim's and Dropout rates and that too without manually rewriting for every layer.

Data Cleaning, Preprocessing is solely inspired from the wonderfully written medium blog that you can check out

*Scoring/eval on the last cell*

Suggestions for improvements are welcomed from every one

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
X=pd.read_csv('/kaggle/input/competitions/asap-aes/training_set_rel3.tsv', sep='\t', encoding='ISO-8859-1')

In [ ]:
X.head()

In [ ]:
y=X['domain1_score']

In [ ]:
X=X['essay']

# **Prepocessing the essay Text to make it ready for Word2Vec and then training**

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from gensim.models import Word2Vec

In [ ]:
###This Function helps in converting the essay into a list of words that is later processed to
### remove stopwords. Output is a 1D list of words in the essay
def essay_to_wordlist(essay,remove_stopwords):
    essay=re.sub("[^a-zA-Z]"," ",essay)
    words=essay.lower().split()
    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words_list=[w for w in words if w not in stops]
    return words_list 

In [ ]:
###Converts essay into a 2D list containing a list of words for each sentence in the essay.Important 
###-for forming the vocabulary of the word2Vec Model
def essay_to_sentences(essay,remove_stopwords):
    tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
    raw_sentences=tokenizer.tokenize(essay.strip())
    sentences=[]
    for raw_sentence in raw_sentences:
        if len(raw_sentence)>0:
            sentences.append(essay_to_wordlist(raw_sentence,remove_stopwords))
    return sentences

In [ ]:
essays=X.values
sentences=[]
for essay in essays:
    sentences.append(essay_to_sentences(essay,remove_stopwords=True))      ###2D list

In [ ]:
###Had to do this because the previous Cell was giving me a 3D list (we need a 2D list)
import itertools
sentences=list(itertools.chain.from_iterable(sentences))      

In [ ]:
embed_dim = 300 
min_word_count = 40
num_workers = 4
context = 10
downsampling = 1e-3

In [ ]:
wv_model= Word2Vec(sentences, workers=num_workers, vector_size=embed_dim, min_count = min_word_count, window = context, sample = downsampling)

In [ ]:
wv_model.init_sims(replace=True)
wv_model.wv.save_word2vec_format('word2vecmodel.bin', binary=True)

In [ ]:
###Train-test-val split on X,y
X=X.sample(frac=1,random_state=42)
y = y.loc[X.index] 
train_ratio=0.8
test_ratio=0.10
train_len=int(train_ratio*len(X))
test_len=int(test_ratio*len(X))
x_train=X[:train_len].values
x_test=X[train_len:train_len+test_len].values
x_val=X[train_len+test_len:].values
y_train=y[:train_len].values
y_test=y[train_len:train_len+test_len].values
y_val=y[train_len+test_len:].values

In [ ]:
def clean_preprocessed_text(x,model,max_len):
    clean_essays = []
    for essay in x:
        clean_essays.append(essay_to_wordlist(essay, remove_stopwords=True))
    vocab_size=len(model.wv.key_to_index)+2       ###Reserved for Padding and OOV tokens
    x_out=[]
    for essay in clean_essays:
        essay_vec=[]
        for word in essay[:max_len]:
            if word in model.wv.key_to_index:
                essay_vec.append(model.wv.key_to_index[word]+2)
            else:
                essay_vec.append(1)
        while len(essay_vec)<max_len:
            essay_vec.append(0)
        x_out.append(essay_vec)
    x_out=np.array(x_out,dtype=np.int32)

    return x_out

In [ ]:
x_train_pad=clean_preprocessed_text(x_train,wv_model,max_len=200)
x_test_pad=clean_preprocessed_text(x_test,wv_model,max_len=200)
x_val_pad=clean_preprocessed_text(x_val,wv_model,max_len=200)

In [ ]:
vocab_size=len(wv_model.wv.key_to_index)+2
embedding_matrix = np.zeros((vocab_size, embed_dim))
for word,idx in wv_model.wv.key_to_index.items():
    embedding_matrix[idx+2]=wv_model.wv[word]

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset

In [ ]:
class CustomDataset(Dataset):
    def __init__(self,x,y):
        self.x=torch.tensor(x,dtype=torch.long)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self):
        return len(self.x)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]

In [ ]:
batch_size=64
train_dataset=CustomDataset(x_train_pad,y_train)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=0)

In [ ]:
batch_size=64
val_dataset=CustomDataset(x_val_pad,y_val)
val_loader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,num_workers=0)

# **Model Architecture Part**


***I use the part down below as a boilerplate template because it's a very easy to build a deep customizable LSTM model with varying dropout rates,dimensions etc. Very scalable and quite reproducable on other type of RNN models as well.***

In [ ]:
class LSTMSequentialWrapper(nn.Module):
    def __init__(self, input_size, hidden_size,dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout=nn.Dropout(dropout) if dropout>0.0 else nn.Identity()
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.dropout(out)

In [ ]:
class manual_stacked_model(nn.Module):
  def __init__(self,embedding_matrix,embed_dim):
    super().__init__()
    weights=torch.FloatTensor(embedding_matrix)
    self.network=nn.Sequential(
    nn.Embedding.from_pretrained(weights,freeze=True),
    LSTMSequentialWrapper(input_size=embed_dim, hidden_size=128,dropout=0.4),
    LSTMSequentialWrapper(128,64,dropout=0.4)
)
    self.fc=nn.Linear(64,1)
  def forward(self,x):
    lstm_output=self.network(x)
    return torch.nn.functional.relu(self.fc(lstm_output[:,-1,:]))

In [ ]:
model=manual_stacked_model(embedding_matrix,embed_dim)

# **Model Training Part**

In [ ]:
def calc_loss_total(data_loader,model,device,num_batches=None):
  total_loss=0
  if len(data_loader)==0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)
      loss=torch.nn.functional.l1_loss(output_batch.squeeze(-1),target_batch)
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
optimizer=torch.optim.RMSprop(model.parameters(),lr=1e-3)
num_epochs=50
eval_freq=50
eval_iter=10

In [ ]:
def train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter):
  global_step=-1
  train_losses=[]
  val_losses=[]
  model.to(device)
  for i in range(num_epochs):
    model.train()
    for i,(input_batch,target_batch) in enumerate(train_loader):
      input_batch,target_batch=input_batch.to(device),target_batch.to(device)
      output_batch=model(input_batch)
      criterion=torch.nn.functional.l1_loss(output_batch.squeeze(-1),target_batch)
      criterion.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
      optimizer.step()
      optimizer.zero_grad()
      global_step+=1
      if global_step%(eval_freq)==0:
        with torch.no_grad():
            model.eval()
            train_loss=calc_loss_total(train_loader,model=model,device=device,num_batches=eval_iter)
            val_loss=calc_loss_total(val_loader,model=model,device=device,num_batches=eval_iter)
            model.train()
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Ep {i+1} (step {global_step:06d}):" f" Train loss:{train_loss:.3f}, Val loss:{val_loss:.3f}")
  return train_losses,val_losses

In [ ]:
train_losses,val_losses=train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(
        train_losses,
        label="train_loss"
    )

plt.plot(
        val_losses,
        label="validation_loss"
    )

plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()

# **Evaluvation on Test data**

In [ ]:
batch_size=64
test_dataset=CustomDataset(x_test_pad,y_test)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=True,num_workers=0)

In [ ]:
from sklearn.metrics import cohen_kappa_score

In [ ]:
model.eval()
with torch.no_grad():
    mean_score=0
    for input_batch,output_batch in test_loader:
        input_batch,output_batch=input_batch.to(device),output_batch.to(device)
        y_pred=model(input_batch)
        y_pred=torch.round(y_pred).int()
        mean_score+=cohen_kappa_score(output_batch.cpu().numpy(),y_pred.cpu().numpy().squeeze(-1),weights='quadratic')
    mean_score/=len(test_loader)
    print("Kappa Score: {}".format(mean_score))

In [ ]:
####There we go, our Kappa Score for test data is 0.9625 that beats the other benchmark results 